# apply, map y transform en pandas

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/14_pandas_transformaciones/code/01_apply_map_transform.ipynb)

Este notebook cubre las funciones que más confusión generan en pandas: `map`, `apply`, `transform` y `pipe`. La confusión viene de que tienen nombres similares, hacen cosas parecidas, pero **lo que recibe `f` es completamente diferente en cada caso**. Si no tienes claro eso, no puedes predecir qué va a salir.

El objetivo es que al terminar tengas un modelo mental claro:

```
¿Qué recibe f()?
──────────────────────────────────────────────────────
Series.map(f)              → 1 escalar (valor de celda)
Series.apply(f)            → 1 escalar (valor de celda)

DataFrame.apply(f, axis=0) → 1 Series (columna completa)
DataFrame.apply(f, axis=1) → 1 Series (fila completa)

DataFrame.map(f)           → 1 escalar (celda a celda)
[antes: DataFrame.applymap en pandas < 2.1]

DataFrame.pipe(f)          → el DataFrame entero
```

Guarda este modelo mental. Lo vamos a construir paso a paso con ejemplos ejecutables.

In [1]:
%pip install pandas==2.2.3 numpy==1.26.4

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np

print(f"pandas: {pd.__version__}")
print(f"numpy:  {np.__version__}")

# Dataset de empleados — lo usaremos en todo el notebook
df = pd.DataFrame({
    "nombre":  ["Ana", "Carlos", "Maria", "Pedro", "Sofia", "Luis", "Elena", "Tomas"],
    "edad":    [25, 32, 28, 45, 22, 38, 31, 27],
    "salario": [35000.0, 52000.0, 41000.0, 68000.0, 29000.0, 55000.0, 47000.0, 33000.0],
    "ciudad":  ["CDMX", "MTY", "CDMX", "GDL", "MTY", "CDMX", "GDL", "MTY"],
    "activo":  [True, True, False, True, True, False, True, True],
})

df

pandas: 2.2.3
numpy:  1.26.4


,nombre,edad,salario,ciudad,activo
0,Ana,25,35000.0,CDMX,True
1,Carlos,32,52000.0,MTY,True
2,Maria,28,41000.0,CDMX,False
3,Pedro,45,68000.0,GDL,True
4,Sofia,22,29000.0,MTY,True
5,Luis,38,55000.0,CDMX,False
6,Elena,31,47000.0,GDL,True
7,Tomas,27,33000.0,MTY,True


---
## Sección 1: `Series.map()` — el más simple

**¿Qué recibe `f`?** Un escalar — el valor de UNA celda.  
**¿Cuántas veces se llama `f`?** Una vez por elemento de la Serie.  
**¿Qué devuelve?** Una nueva Serie del mismo tamaño.

Tres formas de usar `map`: con función, con diccionario, con otra Serie.

In [3]:
# Forma 1: con función lambda
# f recibe cada edad (int), devuelve un int
df["edad"].map(lambda x: x * 2)

0    50
1    64
2    56
3    90
4    44
5    76
6    62
7    54
Name: edad, dtype: int64

In [4]:
# Forma 2: con diccionario (tabla de lookup — muy útil en datos reales)
# f es el dict: busca cada valor y devuelve el correspondiente
mapping = {"CDMX": "Centro", "MTY": "Norte", "GDL": "Occidente"}
df["region"] = df["ciudad"].map(mapping)

df[["ciudad", "region"]]

,ciudad,region
0,CDMX,Centro
1,MTY,Norte
2,CDMX,Centro
3,GDL,Occidente
4,MTY,Norte
5,CDMX,Centro
6,GDL,Occidente
7,MTY,Norte


**Caso edge importante:** ¿Qué pasa si un valor no está en el diccionario?

In [5]:
# Dict incompleto — "MTY" no aparece
mapping_incompleto = {"CDMX": "Centro", "GDL": "Occidente"}
df["ciudad"].map(mapping_incompleto)

0       Centro
1          NaN
2       Centro
3    Occidente
4          NaN
5       Centro
6    Occidente
7          NaN
Name: ciudad, dtype: object

Las ciudades que no están en el dict se vuelven `NaN`. Esto es silencioso — pandas no lanza error. **Siempre verifica que tu mapping tenga todos los valores antes de usarlo.**

In [6]:
# Forma 3: con una Serie (index-to-index lookup)
# map busca cada valor de la Serie original en el INDEX de la Serie argumento
codigo_ciudad = pd.Series(
    {"CDMX": "MX-CMX", "MTY": "MX-NLE", "GDL": "MX-JAL"}
)
df["ciudad"].map(codigo_ciudad)

0    MX-CMX
1    MX-NLE
2    MX-CMX
3    MX-JAL
4    MX-NLE
5    MX-CMX
6    MX-JAL
7    MX-NLE
Name: ciudad, dtype: object

**Comportamiento con NaN:** `map` propaga NaN automáticamente — si el input es NaN, el output es NaN sin llamar a `f`.

In [7]:
# NaN entra → NaN sale, f no se llama para esos valores
s_con_nan = pd.Series([10.0, np.nan, 30.0, np.nan, 50.0])
s_con_nan.map(lambda x: x * 2)

0     20.0
1      NaN
2     60.0
3      NaN
4    100.0
dtype: float64

**💡 Prueba esto:** Intenta `map` con un diccionario que tenga valores duplicados en el output. ¿Funciona? ¿Qué pasa si el dict tiene más claves de las que existen en la Serie?

In [8]:
s = pd.Series(["a", "b", "c", "a"])

mapeo = {
    "a": "vocal",
    "b": "consonante",
    "c": "consonante",
    "d": "extra"
}

resultado = s.map(mapeo)

print("Serie original:")
print(s)

print("\nResultado de map:")
print(resultado)

Serie original:
0    a
1    b
2    c
3    a
dtype: object

Resultado de map:
0         vocal
1    consonante
2    consonante
3         vocal
dtype: object


---
## Sección 2: `Series.apply()` — casi igual a map, con diferencias clave

**¿Qué recibe `f`?** Un escalar — igual que `map`.  
**Diferencias importantes:**
1. `apply` **no** propaga NaN automáticamente — `f` se llama aunque el valor sea NaN
2. `apply` acepta `args=` y `**kwargs` para pasar argumentos extra
3. `apply` solo funciona con funciones, no con dicts ni Series

In [9]:
# Comparación directa: apply vs map con NaN
s_con_nan = pd.Series([1.0, np.nan, 3.0, np.nan])

print("map con multiplicación:")
print(s_con_nan.map(lambda x: x * 2))

print("\napply con multiplicación:")
print(s_con_nan.apply(lambda x: x * 2))
# Aquí son iguales porque x * 2 con NaN da NaN de todas formas

map con multiplicación:
0    2.0
1    NaN
2    6.0
3    NaN
dtype: float64

apply con multiplicación:
0    2.0
1    NaN
2    6.0
3    NaN
dtype: float64


In [12]:
# La diferencia real aparece cuando f fallaría con NaN
# Esta función falla si x es NaN porque int(NaN) lanza ValueError
f_peligrosa = lambda x: len(str(int(x)))

# print("map: NaN se propaga silenciosamente, f no se llama")
# print(s_con_nan.map(f_peligrosa))

f_segura = lambda x: x * 2

print(s_con_nan.map(f_segura))

0    2.0
1    NaN
2    6.0
3    NaN
dtype: float64


In [13]:
print("apply: f sí se llama con NaN → error")
try:
    s_con_nan.apply(f_peligrosa)
except Exception as e:
    print(f"{type(e).__name__}: {e}")

apply: f sí se llama con NaN → error
ValueError: cannot convert float NaN to integer


**Regla:** Si tu función podría fallar con NaN y no quieres manejar ese caso explícitamente, usa `map`. Si necesitas manejar NaN de forma especial dentro de `f`, usa `apply`.

In [14]:
# apply con args= para pasar argumentos posicionales extra
def clasificar_por_umbral(valor, umbral_bajo, umbral_alto):
    """f recibe: el escalar, más argumentos extra que pasamos con args="""
    if valor < umbral_bajo:
        return "bajo"
    elif valor < umbral_alto:
        return "medio"
    else:
        return "alto"

df["nivel_salario"] = df["salario"].apply(
    clasificar_por_umbral, args=(35000, 55000)
)
df[["nombre", "salario", "nivel_salario"]]

,nombre,salario,nivel_salario
0,Ana,35000.0,medio
1,Carlos,52000.0,medio
2,Maria,41000.0,medio
3,Pedro,68000.0,alto
4,Sofia,29000.0,bajo
5,Luis,55000.0,alto
6,Elena,47000.0,medio
7,Tomas,33000.0,bajo


**💡 Prueba esto:** ¿Por qué `map` no acepta `args=`? Intenta `df["salario"].map(clasificar_por_umbral, args=(35000, 55000))` y observa el error.

In [16]:
# df["salario"].map(clasificar_por_umbral, args = (35000, 55000))


---
## Sección 3: `DataFrame.apply(axis=0)` — función sobre columnas

**¿Qué recibe `f`?** Una Serie completa (una columna entera).  
**¿Cuántas veces se llama `f`?** Una vez por columna.  
**`axis=0` es el default.**

Usamos una función "espía" para mostrar exactamente qué recibe `f`:

In [17]:
def spy_columna(col):
    """Esta función muestra qué recibe y luego hace algo útil."""
    print(f"  Recibí: type={type(col).__name__}, nombre='{col.name}', len={len(col)}")
    print(f"  Primeros valores: {col.values[:3]}")
    return col.sum()  # devuelve UN escalar

print("Llamadas a spy_columna:")
resultado = df[["edad", "salario"]].apply(spy_columna, axis=0)
print("\nResultado final:")
print(resultado)

Llamadas a spy_columna:
  Recibí: type=Series, nombre='edad', len=8
  Primeros valores: [25 32 28]
  Recibí: type=Series, nombre='salario', len=8
  Primeros valores: [35000. 52000. 41000.]

Resultado final:
edad          248.0
salario    360000.0
dtype: float64


In [18]:
# Uso real: estadísticas por columna
# f devuelve un escalar → resultado es una Serie (una fila por columna)
df[["edad", "salario"]].apply(lambda col: col.max() - col.min())

edad          23.0
salario    39000.0
dtype: float64

In [19]:
# Normalizar cada columna numérica (z-score)
# f devuelve una Serie del mismo tamaño → resultado es un DataFrame
df[["edad", "salario"]].apply(lambda col: (col - col.mean()) / col.std())

,edad,salario
0,-0.805906,-0.76440
1,0.134318,0.53508
2,-0.402953,-0.30576
3,1.880447,1.75812
4,-1.208859,-1.22304
5,0.940224,0.76440
6,0.000000,0.15288
7,-0.537271,-0.91728


**💡 Prueba esto:** Aplica `spy_columna` sobre `df` completo (incluyendo columnas string). ¿Qué pasa con la suma de strings? ¿Pandas lanza error o lo maneja?

In [20]:
# Tu experimento aquí
df.apply(spy_columna)

  Recibí: type=Series, nombre='nombre', len=8
  Primeros valores: ['Ana' 'Carlos' 'Maria']
  Recibí: type=Series, nombre='edad', len=8
  Primeros valores: [25 32 28]
  Recibí: type=Series, nombre='salario', len=8
  Primeros valores: [35000. 52000. 41000.]
  Recibí: type=Series, nombre='ciudad', len=8
  Primeros valores: ['CDMX' 'MTY' 'CDMX']
  Recibí: type=Series, nombre='activo', len=8
  Primeros valores: [ True  True False]
  Recibí: type=Series, nombre='region', len=8
  Primeros valores: ['Centro' 'Norte' 'Centro']
  Recibí: type=Series, nombre='nivel_salario', len=8
  Primeros valores: ['medio' 'medio' 'medio']


nombre                      AnaCarlosMariaPedroSofiaLuisElenaTomas
edad                                                           248
salario                                                   360000.0
ciudad                                 CDMXMTYCDMXGDLMTYCDMXGDLMTY
activo                                                           6
region           CentroNorteCentroOccidenteNorteCentroOccidente...
nivel_salario                 mediomediomedioaltobajoaltomediobajo
dtype: object

---
## Sección 4: `DataFrame.apply(axis=1)` — función sobre filas

**¿Qué recibe `f`?** Una Serie completa (una fila entera, con el índice siendo los nombres de columnas).  
**¿Cuántas veces se llama `f`?** Una vez por fila.

Este es el caso de uso más común: combinar múltiples columnas en un valor.

In [21]:
# Función espía para filas
contador = [0]

def spy_fila(fila):
    if contador[0] < 2:  # solo mostramos las primeras 2 filas para no saturar la salida
        print(f"  Fila índice={fila.name}: {fila.to_dict()}")
    contador[0] += 1
    return fila["salario"] / fila["edad"]  # valor compuesto usando múltiples columnas

print("Primeras 2 llamadas a spy_fila:")
df["salario_por_edad"] = df.apply(spy_fila, axis=1)
df[["nombre", "edad", "salario", "salario_por_edad"]].head()

Primeras 2 llamadas a spy_fila:
  Fila índice=0: {'nombre': 'Ana', 'edad': 25, 'salario': 35000.0, 'ciudad': 'CDMX', 'activo': True, 'region': 'Centro', 'nivel_salario': 'medio'}
  Fila índice=1: {'nombre': 'Carlos', 'edad': 32, 'salario': 52000.0, 'ciudad': 'MTY', 'activo': True, 'region': 'Norte', 'nivel_salario': 'medio'}


,nombre,edad,salario,salario_por_edad
0,Ana,25,35000.0,1400.000000
1,Carlos,32,52000.0,1625.000000
2,Maria,28,41000.0,1464.285714
3,Pedro,45,68000.0,1511.111111
4,Sofia,22,29000.0,1318.181818


**Por qué es lento `axis=1`:** es un loop de Python — llama a `f` una vez por cada fila. Para DataFrames grandes, esto es mucho más lento que operaciones vectorizadas.

In [22]:
# Comparación de velocidad — creamos un DataFrame más grande para que se note
df_grande = pd.DataFrame({
    "salario": np.random.uniform(25000, 80000, 10_000),
    "activo":  np.random.choice([True, False], 10_000),
})

print("Lento — apply axis=1 (loop Python):")
%timeit df_grande.apply(lambda row: row["salario"] * 1.1 if row["activo"] else row["salario"], axis=1)

print("\nRápido — operaciones vectorizadas pandas:")
%timeit df_grande["salario"] * df_grande["activo"].map({True: 1.1, False: 1.0})

print("\nMás rápido — numpy where:")
%timeit np.where(df_grande["activo"], df_grande["salario"] * 1.1, df_grande["salario"])

Lento — apply axis=1 (loop Python):
52.2 ms ± 1.7 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)

Rápido — operaciones vectorizadas pandas:
237 μs ± 41.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)

Más rápido — numpy where:
88.2 μs ± 4.34 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


**Regla práctica:** Si puedes expresar la operación usando operaciones de columna (aritmética, `np.where`, `pd.cut`), hazlo. Usa `axis=1` solo cuando necesitas lógica compleja que accede a múltiples columnas con condiciones que no se pueden vectorizar fácilmente.

**💡 Prueba esto:** Reescribe el cálculo `salario_por_edad` sin usar `apply`. Pista: puedes dividir dos columnas directamente.

In [23]:
# Tu solución vectorizada aquí
df["salario_por_edad"] = df["salario"] / df["edad"]

df

,nombre,edad,salario,ciudad,activo,region,nivel_salario,salario_por_edad
0,Ana,25,35000.0,CDMX,True,Centro,medio,1400.000000
1,Carlos,32,52000.0,MTY,True,Norte,medio,1625.000000
2,Maria,28,41000.0,CDMX,False,Centro,medio,1464.285714
3,Pedro,45,68000.0,GDL,True,Occidente,alto,1511.111111
4,Sofia,22,29000.0,MTY,True,Norte,bajo,1318.181818
5,Luis,38,55000.0,CDMX,False,Centro,alto,1447.368421
6,Elena,31,47000.0,GDL,True,Occidente,medio,1516.129032
7,Tomas,27,33000.0,MTY,True,Norte,bajo,1222.222222


---
## Sección 5: `DataFrame.map()` — celda a celda

**¿Qué recibe `f`?** Un escalar (el valor de UNA celda).  
**¿Cuántas veces se llama `f`?** Una vez por cada celda del DataFrame.  
**Nota de versión:** se renombró de `applymap` a `map` en pandas 2.1. Si usas pandas < 2.1, usa `.applymap()`.

Úsalo para transformar uniformemente todos los valores de un DataFrame.

In [24]:
# Versión de pandas — importante saber cuál tienes
print(f"pandas version: {pd.__version__}")
print("Usa .map() si >= 2.1, .applymap() si < 2.1")

pandas version: 2.2.3
Usa .map() si >= 2.1, .applymap() si < 2.1


In [25]:
# Redondear todos los números a miles
df_numerico = df[["edad", "salario"]].copy()

print("Original:")
print(df_numerico)

print("\nRedondeado a miles (ndigits=-3):")
df_numerico.map(lambda x: round(x, -3))

Original:
   edad  salario
0    25  35000.0
1    32  52000.0
2    28  41000.0
3    45  68000.0
4    22  29000.0
5    38  55000.0
6    31  47000.0
7    27  33000.0

Redondeado a miles (ndigits=-3):


,edad,salario
0,0,35000.0
1,0,52000.0
2,0,41000.0
3,0,68000.0
4,0,29000.0
5,0,55000.0
6,0,47000.0
7,0,33000.0


In [26]:
# Formatear todos los valores como strings para display
df[["edad", "salario"]].map(lambda x: f"{x:,.1f}")

,edad,salario
0,25.0,"35,000.0"
1,32.0,"52,000.0"
2,28.0,"41,000.0"
3,45.0,"68,000.0"
4,22.0,"29,000.0"
5,38.0,"55,000.0"
6,31.0,"47,000.0"
7,27.0,"33,000.0"


In [27]:
# Detectar si hay algún valor mayor a un umbral (celda a celda)
umbral = 35
df[["edad", "salario"]].map(lambda x: x > umbral)

,edad,salario
0,False,True
1,False,True
2,False,True
3,True,True
4,False,True
5,True,True
6,False,True
7,False,True


**💡 Prueba esto:** ¿Cuál es la diferencia entre `df.map(f)` y `df.apply(lambda col: col.map(f), axis=0)`? ¿Son equivalentes? ¿En qué casos divergirían?

In [29]:
# Tu experimento aquí
f = lambda x: x * 2
df.map(f)
df.apply(lambda col: col.map(f), axis=0)

,nombre,edad,salario,ciudad,activo,region,nivel_salario,salario_por_edad
0,AnaAna,50,70000.0,CDMXCDMX,2,CentroCentro,mediomedio,2800.000000
1,CarlosCarlos,64,104000.0,MTYMTY,2,NorteNorte,mediomedio,3250.000000
2,MariaMaria,56,82000.0,CDMXCDMX,0,CentroCentro,mediomedio,2928.571429
3,PedroPedro,90,136000.0,GDLGDL,2,OccidenteOccidente,altoalto,3022.222222
4,SofiaSofia,44,58000.0,MTYMTY,2,NorteNorte,bajobajo,2636.363636
5,LuisLuis,76,110000.0,CDMXCDMX,0,CentroCentro,altoalto,2894.736842
6,ElenaElena,62,94000.0,GDLGDL,2,OccidenteOccidente,mediomedio,3032.258065
7,TomasTomas,54,66000.0,MTYMTY,2,NorteNorte,bajobajo,2444.444444


---
## Sección 6: El output importa — qué devuelve `f` cambia el resultado

Esta es la parte más confusa. Con `DataFrame.apply`, **el tipo y forma de lo que devuelve `f` determina qué tipo obtienes de vuelta.**

In [30]:
# Caso 1: f devuelve UN escalar → resultado es una Serie (un valor por columna)
resultado = df[["edad", "salario"]].apply(lambda col: col.max())
print("Tipo:", type(resultado).__name__)
print(resultado)

Tipo: Series
edad          45.0
salario    68000.0
dtype: float64


In [31]:
# Caso 2: f devuelve una Serie con el MISMO índice → resultado es un DataFrame
resultado = df[["edad", "salario"]].apply(lambda col: (col - col.mean()) / col.std())
print("Tipo:", type(resultado).__name__)
resultado

Tipo: DataFrame


,edad,salario
0,-0.805906,-0.76440
1,0.134318,0.53508
2,-0.402953,-0.30576
3,1.880447,1.75812
4,-1.208859,-1.22304
5,0.940224,0.76440
6,0.000000,0.15288
7,-0.537271,-0.91728


In [32]:
# Caso 3: f devuelve una Serie con índice DISTINTO → DataFrame con las nuevas etiquetas
resultado = df[["edad", "salario"]].apply(
    lambda col: pd.Series({"maximo": col.max(), "minimo": col.min(), "rango": col.max() - col.min()})
)
print("Tipo:", type(resultado).__name__)
resultado

Tipo: DataFrame


,edad,salario
maximo,45,68000.0
minimo,22,29000.0
rango,23,39000.0


In [33]:
# Caso 4: f devuelve tuple/list con axis=1 y result_type="expand" → columnas separadas
# Sin result_type="expand", cada fila sería una tupla en una sola columna
resultado = df.apply(
    lambda row: (row["nombre"].upper(), len(row["nombre"])),
    axis=1,
    result_type="expand"
)
resultado.columns = ["nombre_upper", "largo_nombre"]
resultado

,nombre_upper,largo_nombre
0,ANA,3
1,CARLOS,6
2,MARIA,5
3,PEDRO,5
4,SOFIA,5
5,LUIS,4
6,ELENA,5
7,TOMAS,5


In [34]:
# Comparación: sin result_type="expand"
sin_expand = df.apply(
    lambda row: (row["nombre"].upper(), len(row["nombre"])),
    axis=1
)
print("Sin expand — cada elemento es una tupla:")
print(sin_expand)

Sin expand — cada elemento es una tupla:
0       (ANA, 3)
1    (CARLOS, 6)
2     (MARIA, 5)
3     (PEDRO, 5)
4     (SOFIA, 5)
5      (LUIS, 4)
6     (ELENA, 5)
7     (TOMAS, 5)
dtype: object


**💡 Prueba esto:** Modifica el Caso 3 para que `f` devuelva solo 2 estadísticas para la columna `edad` pero 4 para `salario`. ¿Qué hace pandas con las celdas faltantes?

In [35]:
# Tu experimento aquí
resultado = df[["edad", "salario"]].apply(
    lambda col: pd.Series(
        {
            "maximo": col.max(),
            "minimo": col.min(),
            **({"rango": col.max() - col.min(), "media": col.mean()} if col.name == "salario" else {})
        }
    )
)

resultado

,edad,salario
maximo,45.0,68000.0
media,NaN,45000.0
minimo,22.0,29000.0
rango,NaN,39000.0


---
## Sección 7: Pasar argumentos extra a `f`

Tres patrones para cuando `f` necesita más que el valor de la celda.

In [36]:
# Forma 1: lambda que captura variables del scope externo
umbral = 40000
df["es_senior"] = df["salario"].apply(lambda x: x > umbral)
df[["nombre", "salario", "es_senior"]]

,nombre,salario,es_senior
0,Ana,35000.0,False
1,Carlos,52000.0,True
2,Maria,41000.0,True
3,Pedro,68000.0,True
4,Sofia,29000.0,False
5,Luis,55000.0,True
6,Elena,47000.0,True
7,Tomas,33000.0,False


In [37]:
# Forma 2: args=() para argumentos posicionales (más explícito, más reutilizable)
def escalar_valor(valor, factor, decimales=0):
    """f(valor, factor, decimales) — los extra vienen de args= y kwargs"""
    return round(valor * factor, decimales)

df["bono"] = df["salario"].apply(escalar_valor, args=(0.15,), decimales=2)
df[["nombre", "salario", "bono"]]

,nombre,salario,bono
0,Ana,35000.0,5250.0
1,Carlos,52000.0,7800.0
2,Maria,41000.0,6150.0
3,Pedro,68000.0,10200.0
4,Sofia,29000.0,4350.0
5,Luis,55000.0,8250.0
6,Elena,47000.0,7050.0
7,Tomas,33000.0,4950.0


In [38]:
# Forma 3: usar otra columna del DataFrame como argumento — requiere axis=1
# No hay forma de pasar "la otra columna" directamente a Series.apply
# Cuando necesitas datos de otra columna, tienes que subir a DataFrame.apply(axis=1)

factor_por_activo = {True: 1.20, False: 1.05}  # 20% más si activo, 5% si no

df["salario_ajustado"] = df.apply(
    lambda row: row["salario"] * factor_por_activo[row["activo"]],
    axis=1
)
df[["nombre", "salario", "activo", "salario_ajustado"]]

,nombre,salario,activo,salario_ajustado
0,Ana,35000.0,True,42000.0
1,Carlos,52000.0,True,62400.0
2,Maria,41000.0,False,43050.0
3,Pedro,68000.0,True,81600.0
4,Sofia,29000.0,True,34800.0
5,Luis,55000.0,False,57750.0
6,Elena,47000.0,True,56400.0
7,Tomas,33000.0,True,39600.0


In [39]:
# Pero recuerda: si puedes vectorizar, es mejor
# La versión vectorizada equivalente al ejemplo anterior:
df["salario_ajustado_v2"] = df["salario"] * df["activo"].map({True: 1.20, False: 1.05})

# Verificar que son iguales
iguales = np.allclose(df["salario_ajustado"], df["salario_ajustado_v2"])
print(f"¿Son iguales? {iguales}")

¿Son iguales? True


**💡 Prueba esto:** Implementa la misma lógica del `salario_ajustado` usando `np.where`. ¿Cuál es más legible? ¿Cuál es más rápido?

In [40]:
# Tu solución aquí
df["salario_ajustado_np"] = np.where(
    df["activo"],
    df["salario"] * 1.1,
    df["salario"]
)

df

,nombre,edad,salario,ciudad,activo,region,nivel_salario,salario_por_edad,es_senior,bono,salario_ajustado,salario_ajustado_v2,salario_ajustado_np
0,Ana,25,35000.0,CDMX,True,Centro,medio,1400.000000,False,5250.0,42000.0,42000.0,38500.0
1,Carlos,32,52000.0,MTY,True,Norte,medio,1625.000000,True,7800.0,62400.0,62400.0,57200.0
2,Maria,28,41000.0,CDMX,False,Centro,medio,1464.285714,True,6150.0,43050.0,43050.0,41000.0
3,Pedro,45,68000.0,GDL,True,Occidente,alto,1511.111111,True,10200.0,81600.0,81600.0,74800.0
4,Sofia,22,29000.0,MTY,True,Norte,bajo,1318.181818,False,4350.0,34800.0,34800.0,31900.0
5,Luis,38,55000.0,CDMX,False,Centro,alto,1447.368421,True,8250.0,57750.0,57750.0,55000.0
6,Elena,31,47000.0,GDL,True,Occidente,medio,1516.129032,True,7050.0,56400.0,56400.0,51700.0
7,Tomas,27,33000.0,MTY,True,Norte,bajo,1222.222222,False,4950.0,39600.0,39600.0,36300.0


---
## Sección 8: `transform()` — mismo tamaño, siempre

**¿Qué recibe `f`?** Una Serie (igual que `apply`).  
**Restricción crítica:** `f` DEBE devolver algo del mismo tamaño que la entrada.  
**Por qué existe:** para usar con `groupby` — agregar estadísticas de grupo sin colapsar las filas.

Este es el patrón más importante de esta sección para feature engineering.

In [41]:
# Problema: quiero agregar el salario promedio de la ciudad de cada empleado

# Con apply: colapsa → solo 3 filas (una por ciudad)
print("groupby + apply → colapsa las filas:")
print(df.groupby("ciudad")["salario"].apply(np.mean))

groupby + apply → colapsa las filas:
ciudad
CDMX    43666.666667
GDL     57500.000000
MTY     38000.000000
Name: salario, dtype: float64


In [42]:
# Con transform: mantiene el tamaño original → cada fila conserva el promedio de SU ciudad
df["salario_medio_ciudad"] = df.groupby("ciudad")["salario"].transform(np.mean)

print("groupby + transform → mantiene todas las filas:")
df[["nombre", "ciudad", "salario", "salario_medio_ciudad"]]

groupby + transform → mantiene todas las filas:


/var/folders/m8/ddjl62rx6015fvcr1g9p9nt80000gn/T/ipykernel_64207/1076467447.py:2: FutureWarning: The provided callable <function mean at 0x1213e4cc0> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  df["salario_medio_ciudad"] = df.groupby("ciudad")["salario"].transform(np.mean)


,nombre,ciudad,salario,salario_medio_ciudad
0,Ana,CDMX,35000.0,43666.666667
1,Carlos,MTY,52000.0,38000.000000
2,Maria,CDMX,41000.0,43666.666667
3,Pedro,GDL,68000.0,57500.000000
4,Sofia,MTY,29000.0,38000.000000
5,Luis,CDMX,55000.0,43666.666667
6,Elena,GDL,47000.0,57500.000000
7,Tomas,MTY,33000.0,38000.000000


In [43]:
# Por qué esto es mejor que merge-back (el patrón alternativo)
# Alternativa con merge (más verboso, mismo resultado):
promedios = df.groupby("ciudad")["salario"].mean().reset_index()
promedios.columns = ["ciudad", "salario_medio_ciudad_v2"]
df_alt = df.merge(promedios, on="ciudad")

iguales = np.allclose(df["salario_medio_ciudad"], df_alt["salario_medio_ciudad_v2"])
print(f"¿Resultados iguales? {iguales}")
print("transform es más conciso y legible que groupby + merge")

¿Resultados iguales? True
transform es más conciso y legible que groupby + merge


In [44]:
# Aplicación clásica en ML: diferencia respecto al promedio del grupo
df["desviacion_del_promedio_ciudad"] = df["salario"] - df["salario_medio_ciudad"]
df[["nombre", "ciudad", "salario", "salario_medio_ciudad", "desviacion_del_promedio_ciudad"]]

,nombre,ciudad,salario,salario_medio_ciudad,desviacion_del_promedio_ciudad
0,Ana,CDMX,35000.0,43666.666667,-8666.666667
1,Carlos,MTY,52000.0,38000.000000,14000.000000
2,Maria,CDMX,41000.0,43666.666667,-2666.666667
3,Pedro,GDL,68000.0,57500.000000,10500.000000
4,Sofia,MTY,29000.0,38000.000000,-9000.000000
5,Luis,CDMX,55000.0,43666.666667,11333.333333
6,Elena,GDL,47000.0,57500.000000,-10500.000000
7,Tomas,MTY,33000.0,38000.000000,-5000.000000


In [45]:
# transform también funciona sin groupby, pero es menos común
# Aquí: normalizar la columna salario (z-score) — f recibe toda la Serie, devuelve Serie del mismo tamaño
df[["edad", "salario"]].transform(lambda col: (col - col.mean()) / col.std())

,edad,salario
0,-0.805906,-0.76440
1,0.134318,0.53508
2,-0.402953,-0.30576
3,1.880447,1.75812
4,-1.208859,-1.22304
5,0.940224,0.76440
6,0.000000,0.15288
7,-0.537271,-0.91728


**💡 Prueba esto:** Usa `transform` para agregar el salario máximo por ciudad y el salario mínimo por ciudad, todo en el mismo DataFrame. ¿Puedes hacerlo en dos líneas?

In [46]:
# Tu solución aquí
df["salario_max_ciudad"] = df.groupby("ciudad")["salario"].transform("max")
df["salario_min_ciudad"] = df.groupby("ciudad")["salario"].transform("min")

df

,nombre,edad,salario,ciudad,activo,region,nivel_salario,salario_por_edad,es_senior,bono,salario_ajustado,salario_ajustado_v2,salario_ajustado_np,salario_medio_ciudad,desviacion_del_promedio_ciudad,salario_max_ciudad,salario_min_ciudad
0,Ana,25,35000.0,CDMX,True,Centro,medio,1400.000000,False,5250.0,42000.0,42000.0,38500.0,43666.666667,-8666.666667,55000.0,35000.0
1,Carlos,32,52000.0,MTY,True,Norte,medio,1625.000000,True,7800.0,62400.0,62400.0,57200.0,38000.000000,14000.000000,52000.0,29000.0
2,Maria,28,41000.0,CDMX,False,Centro,medio,1464.285714,True,6150.0,43050.0,43050.0,41000.0,43666.666667,-2666.666667,55000.0,35000.0
3,Pedro,45,68000.0,GDL,True,Occidente,alto,1511.111111,True,10200.0,81600.0,81600.0,74800.0,57500.000000,10500.000000,68000.0,47000.0
4,Sofia,22,29000.0,MTY,True,Norte,bajo,1318.181818,False,4350.0,34800.0,34800.0,31900.0,38000.000000,-9000.000000,52000.0,29000.0
5,Luis,38,55000.0,CDMX,False,Centro,alto,1447.368421,True,8250.0,57750.0,57750.0,55000.0,43666.666667,11333.333333,55000.0,35000.0
6,Elena,31,47000.0,GDL,True,Occidente,medio,1516.129032,True,7050.0,56400.0,56400.0,51700.0,57500.000000,-10500.000000,68000.0,47000.0
7,Tomas,27,33000.0,MTY,True,Norte,bajo,1222.222222,False,4950.0,39600.0,39600.0,36300.0,38000.000000,-5000.000000,52000.0,29000.0


---
## Sección 9: `pipe()` — para el DataFrame completo

**¿Qué recibe `f`?** El DataFrame (o Serie) entero.  
**Para qué sirve:** encadenar funciones personalizadas de forma legible, evitando variables temporales.

Útil cuando tienes transformaciones que aplicas en pipeline.

In [47]:
# Sin pipe: difícil de leer, muchas variables temporales
df_paso1 = df.copy()
df_paso1["salario_norm"] = (df_paso1["salario"] - df_paso1["salario"].mean()) / df_paso1["salario"].std()
df_paso2 = df_paso1.copy()
df_paso2["edad_norm"] = (df_paso2["edad"] - df_paso2["edad"].mean()) / df_paso2["edad"].std()
df_paso2[["nombre", "salario_norm", "edad_norm"]].head(3)

,nombre,salario_norm,edad_norm
0,Ana,-0.76440,-0.805906
1,Carlos,0.53508,0.134318
2,Maria,-0.30576,-0.402953


In [48]:
# Con pipe: encadenamiento limpio
def normalizar_columna(df, columna):
    """f recibe el DataFrame completo + argumentos extra"""
    df = df.copy()
    col = df[columna]
    df[f"{columna}_norm"] = (col - col.mean()) / col.std()
    return df

def filtrar_activos(df):
    """f recibe el DataFrame completo"""
    return df[df["activo"] == True].copy()

resultado = (
    df
    .pipe(normalizar_columna, columna="salario")
    .pipe(normalizar_columna, columna="edad")
    .pipe(filtrar_activos)
)
resultado[["nombre", "activo", "salario_norm", "edad_norm"]]

,nombre,activo,salario_norm,edad_norm
0,Ana,True,-0.76440,-0.805906
1,Carlos,True,0.53508,0.134318
3,Pedro,True,1.75812,1.880447
4,Sofia,True,-1.22304,-1.208859
6,Elena,True,0.15288,0.000000
7,Tomas,True,-0.91728,-0.537271


**💡 Prueba esto:** Agrega una tercera función al pipeline que ordene el DataFrame por `salario_norm` de mayor a menor. Encadénala con `.pipe()`.

In [49]:
# Tu solución aquí
def ordenar_por_salario(df):
    return df.sort_values("salario_norm", ascending=False)


resultado = (
    df
    .pipe(normalizar_columna, columna="salario")
    .pipe(normalizar_columna, columna="edad")
    .pipe(filtrar_activos)
    .pipe(ordenar_por_salario)
)

resultado

,nombre,edad,salario,ciudad,activo,region,nivel_salario,salario_por_edad,es_senior,bono,salario_ajustado,salario_ajustado_v2,salario_ajustado_np,salario_medio_ciudad,desviacion_del_promedio_ciudad,salario_max_ciudad,salario_min_ciudad,salario_norm,edad_norm
3,Pedro,45,68000.0,GDL,True,Occidente,alto,1511.111111,True,10200.0,81600.0,81600.0,74800.0,57500.000000,10500.000000,68000.0,47000.0,1.75812,1.880447
1,Carlos,32,52000.0,MTY,True,Norte,medio,1625.000000,True,7800.0,62400.0,62400.0,57200.0,38000.000000,14000.000000,52000.0,29000.0,0.53508,0.134318
6,Elena,31,47000.0,GDL,True,Occidente,medio,1516.129032,True,7050.0,56400.0,56400.0,51700.0,57500.000000,-10500.000000,68000.0,47000.0,0.15288,0.000000
0,Ana,25,35000.0,CDMX,True,Centro,medio,1400.000000,False,5250.0,42000.0,42000.0,38500.0,43666.666667,-8666.666667,55000.0,35000.0,-0.76440,-0.805906
7,Tomas,27,33000.0,MTY,True,Norte,bajo,1222.222222,False,4950.0,39600.0,39600.0,36300.0,38000.000000,-5000.000000,52000.0,29000.0,-0.91728,-0.537271
4,Sofia,22,29000.0,MTY,True,Norte,bajo,1318.181818,False,4350.0,34800.0,34800.0,31900.0,38000.000000,-9000.000000,52000.0,29000.0,-1.22304,-1.208859


---
## Sección 10: Tabla de decisión — cuándo usar qué

| Quiero... | Usar... | f recibe... |
|-----------|---------|-------------|
| Transformar celda por celda en una Serie | `Series.map(f)` | 1 escalar |
| Lo mismo pero con control sobre NaN o args extra | `Series.apply(f)` | 1 escalar |
| Lookup con diccionario | `Series.map(dict)` | N/A (el dict hace el trabajo) |
| Aplicar función a cada columna | `DataFrame.apply(f, axis=0)` | 1 Serie (columna) |
| Combinar múltiples columnas por fila | `DataFrame.apply(f, axis=1)` | 1 Serie (fila) |
| Transformar cada celda del DataFrame | `DataFrame.map(f)` | 1 escalar |
| Agregar función a un pipeline | `DataFrame.pipe(f)` | el DataFrame entero |
| Estadística de grupo sin colapsar filas | `groupby().transform(f)` | 1 Serie (grupo) |

**Árbol de decisión rápido:**
```
¿Operas sobre una Serie?
  ├── ¿Usas un dict de lookup?          → map(dict)
  ├── ¿Necesitas args= o NaN explícito? → apply()
  └── En cualquier otro caso            → map()

¿Operas sobre un DataFrame?
  ├── ¿Necesitas combinar columnas por fila?     → apply(axis=1) [busca vectorizar]
  ├── ¿Aplicas por columna?                      → apply(axis=0)
  ├── ¿Aplicas celda a celda?                    → map()
  ├── ¿Estadística de grupo sin colapsar?        → groupby().transform()
  └── ¿Encadenando funciones de transformación? → pipe()
```

---
## Sección 11: Ejercicio integrador

Trabajamos con el mismo DataFrame de empleados. Completa cada paso con la función apropiada.

**Objetivo:** construir un DataFrame enriquecido con varias features derivadas.

In [51]:
# Dataset limpio para el ejercicio
df_ej = pd.DataFrame({
    "nombre":  ["Ana", "Carlos", "Maria", "Pedro", "Sofia", "Luis", "Elena", "Tomas"],
    "edad":    [25, 32, 28, 45, 22, 38, 31, 27],
    "salario": [35000.0, 52000.0, 41000.0, 68000.0, 29000.0, 55000.0, 47000.0, 33000.0],
    "ciudad":  ["CDMX", "MTY", "CDMX", "GDL", "MTY", "CDMX", "GDL", "MTY"],
    "activo":  [True, True, False, True, True, False, True, True],
})
df_ej

,nombre,edad,salario,ciudad,activo
0,Ana,25,35000.0,CDMX,True
1,Carlos,32,52000.0,MTY,True
2,Maria,28,41000.0,CDMX,False
3,Pedro,45,68000.0,GDL,True
4,Sofia,22,29000.0,MTY,True
5,Luis,38,55000.0,CDMX,False
6,Elena,31,47000.0,GDL,True
7,Tomas,27,33000.0,MTY,True


In [52]:
# Paso 1: Usar Series.map() con diccionario para crear columna "region"
# CDMX → "Centro", MTY → "Norte", GDL → "Occidente"
mapa_region = {
    "CDMX": "Centro",
    "MTY": "Norte",
    "GDL": "Occidente"
}

df_ej["region"] = df_ej["ciudad"].map(mapa_region)
print(df_ej[["ciudad", "region"]])

  ciudad     region
0   CDMX     Centro
1    MTY      Norte
2   CDMX     Centro
3    GDL  Occidente
4    MTY      Norte
5   CDMX     Centro
6    GDL  Occidente
7    MTY      Norte


In [53]:
# Paso 2: Usar Series.apply() con args para clasificar salario en 3 niveles
# < 35000 → "bajo", 35000-55000 → "medio", > 55000 → "alto"
# REQUISITO: usa la función clasificar_por_umbral definida en sección 2 + args=
df_ej["nivel_salario"] = df_ej["salario"].apply(
    clasificar_por_umbral,
    args=(35000, 55000)
)

print(df_ej[["nombre", "salario", "nivel_salario"]])

   nombre  salario nivel_salario
0     Ana  35000.0         medio
1  Carlos  52000.0         medio
2   Maria  41000.0         medio
3   Pedro  68000.0          alto
4   Sofia  29000.0          bajo
5    Luis  55000.0          alto
6   Elena  47000.0         medio
7   Tomas  33000.0          bajo


In [54]:
# Paso 3: Usar DataFrame.apply(axis=1) para crear "score_compuesto"
# Formula: (salario / edad) + (10 si activo, else 0)
df_ej["score_compuesto"] = df_ej.apply(
    lambda row: (row["salario"] / row["edad"]) + (10 if row["activo"] else 0),
    axis=1
)

print(df_ej[["nombre", "edad", "salario", "activo", "score_compuesto"]])

   nombre  edad  salario  activo  score_compuesto
0     Ana    25  35000.0    True      1410.000000
1  Carlos    32  52000.0    True      1635.000000
2   Maria    28  41000.0   False      1464.285714
3   Pedro    45  68000.0    True      1521.111111
4   Sofia    22  29000.0    True      1328.181818
5    Luis    38  55000.0   False      1447.368421
6   Elena    31  47000.0    True      1526.129032
7   Tomas    27  33000.0    True      1232.222222


In [55]:
# Paso 4: Usar groupby().transform() para agregar el salario promedio de la ciudad
# Columna nueva: "salario_medio_ciudad"
df_ej["salario_medio_ciudad"] = df_ej.groupby("ciudad")["salario"].transform("mean")

print(df_ej[["nombre", "ciudad", "salario", "salario_medio_ciudad"]])

   nombre ciudad  salario  salario_medio_ciudad
0     Ana   CDMX  35000.0          43666.666667
1  Carlos    MTY  52000.0          38000.000000
2   Maria   CDMX  41000.0          43666.666667
3   Pedro    GDL  68000.0          57500.000000
4   Sofia    MTY  29000.0          38000.000000
5    Luis   CDMX  55000.0          43666.666667
6   Elena    GDL  47000.0          57500.000000
7   Tomas    MTY  33000.0          38000.000000


In [56]:
# Paso 5: Reescribir el score_compuesto del Paso 3 usando operaciones vectorizadas
# y comparar velocidad con %timeit
resultado_vectorizado = (df_ej["salario"] / df_ej["edad"]) + (df_ej["activo"] * 10)
df_ej["score_compuesto_v2"] = resultado_vectorizado
resultado_apply = df_ej["score_compuesto"]

iguales = np.allclose(resultado_apply, resultado_vectorizado)

print(f"¿Son iguales? {iguales}")

¿Son iguales? True


### Soluciones (no hagas trampa)

Descomenta la celda siguiente para ver las soluciones:

In [ ]:
# # ---- SOLUCIONES ----

# # Paso 1: map con diccionario
# df_ej["region"] = df_ej["ciudad"].map({"CDMX": "Centro", "MTY": "Norte", "GDL": "Occidente"})

# # Paso 2: apply con args
# def clasificar_por_umbral(valor, umbral_bajo, umbral_alto):
#     if valor < umbral_bajo: return "bajo"
#     elif valor < umbral_alto: return "medio"
#     else: return "alto"
# df_ej["nivel_salario"] = df_ej["salario"].apply(clasificar_por_umbral, args=(35000, 55000))

# # Paso 3: apply axis=1
# df_ej["score_compuesto"] = df_ej.apply(
#     lambda row: (row["salario"] / row["edad"]) + (10 if row["activo"] else 0),
#     axis=1
# )

# # Paso 4: groupby transform
# df_ej["salario_medio_ciudad"] = df_ej.groupby("ciudad")["salario"].transform(np.mean)

# # Paso 5: versión vectorizada del score_compuesto
# score_vectorizado = (df_ej["salario"] / df_ej["edad"]) + np.where(df_ej["activo"], 10, 0)
# print("¿Iguales?", np.allclose(df_ej["score_compuesto"], score_vectorizado))
# %timeit df_ej.apply(lambda row: (row["salario"] / row["edad"]) + (10 if row["activo"] else 0), axis=1)
# %timeit (df_ej["salario"] / df_ej["edad"]) + np.where(df_ej["activo"], 10, 0)

# print(df_ej)

---
## Resumen final

```
¿Qué recibe f()?
──────────────────────────────────────────────────────
Series.map(f)              → 1 escalar (NaN se propaga sin llamar f)
Series.apply(f)            → 1 escalar (f sí se llama con NaN)

DataFrame.apply(f, axis=0) → 1 Serie (columna completa)
DataFrame.apply(f, axis=1) → 1 Serie (fila completa) — LENTO, busca vectorizar

DataFrame.map(f)           → 1 escalar (celda a celda)
[antes: applymap en pandas < 2.1]

groupby().transform(f)     → 1 Serie (por grupo), mismo tamaño que el original

DataFrame.pipe(f)          → el DataFrame entero
```

**El error más común:** usar `apply(axis=1)` cuando se puede vectorizar. Si te encuentras haciendo `df.apply(lambda row: row["a"] + row["b"], axis=1)`, simplemente escribe `df["a"] + df["b"]`.